In [ ]:
from fastapi import FastAPI
from fastapi.testclient import TestClient

app = FastAPI(title="Sentiment Analysis API")

@app.get("/health")#decorator for the health check endpoint
def health_check():
    return {"status": "healthy"}

client = TestClient(app)
response = client.get("/health")
print("status code:", response.status_code)
print("response:", response.json())


status code: 200
response: {'status': 'healthy'}


In [7]:
print("swagger_url:", app.docs_url)
print("openapi_url:", app.openapi_url)

swagger_url: /docs
openapi_url: /openapi.json


In [8]:
from pydantic import BaseModel, Field
from typing import Literal

class PredictRequest(BaseModel):
    text: str = Field(min_length=1, description="Input text for sentiment analysis")

class PredictResponse(BaseModel):
  label: Literal["positive", "negative", "neutral"] = Field(description="Predicted sentiment label")
  score: float = Field(ge=0, le=1, description="Confidence score of the prediction")

def mock_sentiment_analysis(text: str) -> PredictResponse:
    # Mock sentiment analysis logic
    if any(word in text.lower() for word in ["good", "great", "excellent"]):
        return PredictResponse(label="positive", score=0.95)
    elif any(word in text.lower() for word in ["bad", "terrible", "awful"]):
        return PredictResponse(label="negative", score=0.9)
    else:
        return PredictResponse(label="neutral", score=0.5)

@app.post("/predict", response_model=PredictResponse)
def predict(request: PredictRequest):
    prediction = mock_sentiment_analysis(request.text)
    return prediction

client = TestClient(app)
response = client.post("/predict", json={"text": "This is a great product!"})
print("status code:", response.status_code)
print("response:", response.json())

status code: 200
response: {'label': 'positive', 'score': 0.95}


In [9]:
bad = client.post("/predict", json={"text": ""})
print("status code:", bad.status_code)
print("response:", bad.json())

status code: 422
response: {'detail': [{'type': 'string_too_short', 'loc': ['body', 'text'], 'msg': 'String should have at least 1 character', 'input': '', 'ctx': {'min_length': 1}}]}


In [15]:
#Blocking and async code can be used in FastAPI. The following is an example of an async endpoint.
import asyncio, time

async def fake_model_inference(text: str):
    await asyncio.sleep(1)  # Simulate a delay for model inference
    return f"Processed: {text}"

async def runsequential_inference(texts):
  start = time.perf_counter()
  results = []
  for text in range(3):
    results.append(await fake_model_inference(f"Text {text}"))
  elapsed = time.perf_counter() - start
  return results, elapsed

async def runparallel_inference(texts):
  start = time.perf_counter()
  results = await asyncio.gather(*[fake_model_inference(f"Text {text}") for text in range(3)])
  elapsed = time.perf_counter() - start
  return results, elapsed


In [16]:
import threading

def run_async_in_thread(coro):
    asyncio.get_event_loop()

    result = {}
    def worker():
      result['value'] = asyncio.run(coro)
    t = threading.Thread(target=worker)
    t.start()
    t.join() 
    return result['value']

seq_results, seq_elapsed = run_async_in_thread(runsequential_inference(range(3)))
concurrent_results, concurrent_elapsed = run_async_in_thread(runparallel_inference(range(3)))

print(f"Sequential results: {seq_results}, elapsed time: {seq_elapsed:.2f} seconds")
print(f"Concurrent results: {concurrent_results}, elapsed time: {concurrent_elapsed:.2f} seconds")

speedup = seq_elapsed / concurrent_elapsed if concurrent_elapsed > 0 else float('inf')
print(f"Speedup: {speedup:.2f}")

Sequential results: ['Processed: Text 0', 'Processed: Text 1', 'Processed: Text 2'], elapsed time: 3.01 seconds
Concurrent results: ['Processed: Text 0', 'Processed: Text 1', 'Processed: Text 2'], elapsed time: 1.00 seconds
Speedup: 3.01


In [28]:
#write pytest good, bad and edge cases for the predict endpoint. Use the TestClient to make requests to the endpoint and assert the expected responses.
test_code = '''
from fastapi.testclient import TestClient
from ai_fastapi import app

client = TestClient(app)

def test_predict_good():
    response = client.post("/predict", json={"text": "This is a great product!"})
    assert response.status_code == 200
    data = response.json()
    assert data["label"] == "positive"
    assert 0 <= data["score"] <= 1

def test_predict_bad():
    response = client.post("/predict", json={"text": ""})
    assert response.status_code == 422

def test_predict_edge():
    response = client.post("/predict", json={"text": "This is a terrible product!"})
    assert response.status_code == 200
    data = response.json()
    assert data["label"] == "negative"
    assert 0 <= data["score"] <= 1

'''
with open("test_predict.py", "w") as f:
    f.write(test_code)

In [29]:
import subprocess, sys
result = subprocess.run([sys.executable, "-m", "pytest", "test_predict.py"], capture_output=True, text=True)
print(result.stdout)

============================= test session starts ==============================
platform darwin -- Python 3.15.2, pytest-9.2.0, pluggy-1.7.0
rootdir: /Users/shamseer/Development/workspace/ai-training
plugins: platformdirs-4.14.7, anyio-4.16.0
collected 3 items

test_predict.py ...                                                      [100%]

============================== 3 passed in 0.51s ===============================

